# Прогноз покупки клиента в течение 90 дней

**Цель проекта:** предсказать вероятность того, что клиент интернет-магазина совершит покупку в ближайшие 90 дней.

В проекте используются данные о покупках, истории рекламных рассылок и бинарный целевой признак. Основная метрика качества — **ROC-AUC**, так как задача является бинарной классификацией с несбалансированными классами.

## План работы

1. Загрузка и первичный анализ данных.
2. Проверка пропусков, дубликатов и типов данных.
3. Создание признаков по покупкам.
4. Создание признаков по рассылкам с чтением большой таблицы чанками.
5. Объединение признаков с целевым признаком.
6. Обучение нескольких моделей `sklearn`.
7. Подбор гиперпараметров и выбор лучшей модели по ROC-AUC.
8. Финальное тестирование.

In [ ]:
import os
import ast
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import roc_auc_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 120)
RANDOM_STATE = 42

## Загрузка данных

Ноутбук умеет читать данные двумя способами:
- из папки `data/`, если CSV-файлы уже распакованы;
- из архива `data/filtered_data.zip`, если данные лежат в архиве.

In [ ]:
DATA_DIR = Path('data')
ZIP_PATH = DATA_DIR / 'filtered_data.zip'


def read_csv_from_data(filename, **kwargs):
    csv_path = DATA_DIR / filename
    if csv_path.exists():
        return pd.read_csv(csv_path, **kwargs)
    if ZIP_PATH.exists():
        with zipfile.ZipFile(ZIP_PATH) as z:
            matches = [name for name in z.namelist() if name.endswith(filename)]
            if not matches:
                raise FileNotFoundError(f'{filename} не найден в архиве {ZIP_PATH}')
            with z.open(matches[0]) as f:
                return pd.read_csv(f, **kwargs)
    raise FileNotFoundError(f'Не найден файл {filename} ни в {DATA_DIR}, ни в архиве {ZIP_PATH}')

purchases = read_csv_from_data('apparel-purchases.csv')
target = read_csv_from_data('apparel-target_binary.csv')
full_campaign_daily_event = read_csv_from_data('full_campaign_daily_event.csv')
full_campaign_daily_event_channel = read_csv_from_data('full_campaign_daily_event_channel.csv')

print('purchases:', purchases.shape)
print('target:', target.shape)
print('full_campaign_daily_event:', full_campaign_daily_event.shape)
print('full_campaign_daily_event_channel:', full_campaign_daily_event_channel.shape)

In [ ]:
display(purchases.head())
display(target.head())
display(full_campaign_daily_event.head())
display(full_campaign_daily_event_channel.head())

## Первичный анализ данных

In [ ]:
for name, df in {
    'purchases': purchases,
    'target': target,
    'full_campaign_daily_event': full_campaign_daily_event,
    'full_campaign_daily_event_channel': full_campaign_daily_event_channel,
}.items():
    print('
' + '=' * 80)
    print(name, df.shape)
    display(pd.DataFrame({
        'dtype': df.dtypes.astype(str),
        'missing': df.isna().sum(),
        'missing_share': df.isna().mean().round(4),
        'nunique': df.nunique(dropna=False)
    }))
    print('duplicated rows:', df.duplicated().sum())

In [ ]:
print(target['target'].value_counts())
print('
Доли классов:')
print(target['target'].value_counts(normalize=True).round(4))

Классы несбалансированы, поэтому при разбиении используем стратификацию, а для моделей, где это поддерживается, задаём `class_weight='balanced'`.

## Подготовка признаков по покупкам

Для каждого клиента создаём агрегаты:
- количество покупок и товарных строк;
- суммарное количество товаров;
- сумма, средний чек, медиана, максимум и минимум цены;
- количество дней с покупками;
- давность первой и последней покупки относительно максимальной даты в данных;
- признаки по категориям товаров.

Категории обрабатываются как множество идентификаторов, потому что один и тот же ID может встречаться на разных уровнях дерева категорий.

In [ ]:
purchases = purchases.copy()
purchases['date'] = pd.to_datetime(purchases['date'])
purchases['quantity'] = pd.to_numeric(purchases['quantity'], errors='coerce')
purchases['price'] = pd.to_numeric(purchases['price'], errors='coerce')
purchases['revenue'] = purchases['quantity'] * purchases['price']

max_purchase_date = purchases['date'].max()

purchase_features = purchases.groupby('client_id').agg(
    purchase_rows=('price', 'size'),
    purchase_orders=('message_id', 'nunique'),
    total_quantity=('quantity', 'sum'),
    avg_quantity=('quantity', 'mean'),
    total_spent=('revenue', 'sum'),
    avg_price=('price', 'mean'),
    median_price=('price', 'median'),
    max_price=('price', 'max'),
    min_price=('price', 'min'),
    active_purchase_days=('date', 'nunique'),
    first_purchase_date=('date', 'min'),
    last_purchase_date=('date', 'max'),
).reset_index()

purchase_features['days_since_first_purchase'] = (max_purchase_date - purchase_features['first_purchase_date']).dt.days
purchase_features['days_since_last_purchase'] = (max_purchase_date - purchase_features['last_purchase_date']).dt.days
purchase_features['purchase_period_days'] = (
    purchase_features['last_purchase_date'] - purchase_features['first_purchase_date']
).dt.days
purchase_features['avg_order_value'] = purchase_features['total_spent'] / purchase_features['purchase_orders'].replace(0, np.nan)
purchase_features['avg_items_per_order'] = purchase_features['total_quantity'] / purchase_features['purchase_orders'].replace(0, np.nan)

purchase_features = purchase_features.drop(columns=['first_purchase_date', 'last_purchase_date'])
purchase_features.head()

In [ ]:
def parse_categories(value):
    if pd.isna(value):
        return []
    try:
        parsed = ast.literal_eval(value)
        if isinstance(parsed, list):
            return [str(x) for x in parsed]
    except Exception:
        pass
    return [x.strip().strip("'"") for x in str(value).strip('[]').split(',') if x.strip()]

category_lists = purchases[['client_id', 'category_ids']].copy()
category_lists['category_list'] = category_lists['category_ids'].apply(parse_categories)
category_exploded = category_lists[['client_id', 'category_list']].explode('category_list')
category_exploded = category_exploded.dropna(subset=['category_list'])

TOP_N_CATEGORIES = 50
top_categories = category_exploded['category_list'].value_counts().head(TOP_N_CATEGORIES).index
category_top = category_exploded[category_exploded['category_list'].isin(top_categories)]

category_features = (
    pd.crosstab(category_top['client_id'], category_top['category_list'])
    .add_prefix('cat_')
    .reset_index()
)

purchase_features = purchase_features.merge(category_features, on='client_id', how='left')
cat_cols = [c for c in purchase_features.columns if c.startswith('cat_')]
purchase_features[cat_cols] = purchase_features[cat_cols].fillna(0)

purchase_features.head()

## Подготовка признаков по рассылкам

Таблица `apparel-messages.csv` большая, поэтому она читается чанками. На уровне клиента считаются:
- количество событий каждого типа;
- количество событий по каналам;
- число уникальных кампаний и сообщений;
- даты первой и последней активности;
- доли открытий, кликов, покупок, отписок и bounce-событий относительно отправок.

In [ ]:
def iter_messages(chunksize=1_000_000):
    filename = 'apparel-messages.csv'
    csv_path = DATA_DIR / filename
    if csv_path.exists():
        yield from pd.read_csv(csv_path, chunksize=chunksize)
    elif ZIP_PATH.exists():
        with zipfile.ZipFile(ZIP_PATH) as z:
            matches = [name for name in z.namelist() if name.endswith(filename)]
            if not matches:
                raise FileNotFoundError(f'{filename} не найден в архиве {ZIP_PATH}')
            with z.open(matches[0]) as f:
                yield from pd.read_csv(f, chunksize=chunksize)
    else:
        raise FileNotFoundError(f'Не найден файл {filename}')


def build_message_features(chunksize=1_000_000):
    event_parts = []
    channel_parts = []
    base_parts = []
    date_parts = []

    for i, chunk in enumerate(iter_messages(chunksize=chunksize), start=1):
        chunk['date'] = pd.to_datetime(chunk['date'])

        event_counts = pd.crosstab(chunk['client_id'], chunk['event'])
        event_parts.append(event_counts)

        channel_counts = pd.crosstab(chunk['client_id'], chunk['channel'])
        channel_parts.append(channel_counts)

        base = chunk.groupby('client_id').agg(
            message_rows=('message_id', 'size'),
            unique_messages=('message_id', 'nunique'),
            unique_campaigns=('bulk_campaign_id', 'nunique')
        )
        base_parts.append(base)

        dates = chunk.groupby('client_id').agg(
            first_message_date=('date', 'min'),
            last_message_date=('date', 'max')
        )
        date_parts.append(dates)

        print(f'Обработан чанк {i}: {len(chunk):,} строк')

    event_features = pd.concat(event_parts).groupby(level=0).sum().add_prefix('event_')
    channel_features = pd.concat(channel_parts).groupby(level=0).sum().add_prefix('channel_')
    base_features = pd.concat(base_parts).groupby(level=0).sum()
    date_features = pd.concat(date_parts).groupby(level=0).agg({
        'first_message_date': 'min',
        'last_message_date': 'max'
    })

    features = base_features.join([event_features, channel_features, date_features], how='outer').reset_index()
    max_message_date = features['last_message_date'].max()
    features['days_since_first_message'] = (max_message_date - features['first_message_date']).dt.days
    features['days_since_last_message'] = (max_message_date - features['last_message_date']).dt.days
    features['message_period_days'] = (features['last_message_date'] - features['first_message_date']).dt.days
    features = features.drop(columns=['first_message_date', 'last_message_date'])

    event_cols = [c for c in features.columns if c.startswith('event_')]
    channel_cols = [c for c in features.columns if c.startswith('channel_')]
    features[event_cols + channel_cols] = features[event_cols + channel_cols].fillna(0)

    send_col = 'event_send'
    if send_col in features.columns:
        denom = features[send_col].replace(0, np.nan)
        for event in ['open', 'click', 'purchase', 'unsubscribe', 'hard_bounce', 'soft_bounce', 'complain', 'hbq_spam', 'close']:
            col = f'event_{event}'
            if col in features.columns:
                features[f'rate_{event}_to_send'] = features[col] / denom

    return features

message_features = build_message_features(chunksize=1_000_000)
message_features.head()

## Дополнительные признаки по агрегатам рекламных кампаний

Агрегаты общей базы рассылок используются как признаки уровня кампаний. Их нельзя суммировать по `nunique_*` за разные даты, поэтому для таких колонок берём среднее и максимум, а для `count_*` допускаем сумму.

In [ ]:
def build_campaign_features(df, prefix):
    df = df.copy()
    count_cols = [c for c in df.columns if c.startswith('count_')]
    nunique_cols = [c for c in df.columns if c.startswith('nunique_')]
    agg = {}
    for col in count_cols:
        agg[col] = ['sum', 'mean', 'max']
    for col in nunique_cols:
        agg[col] = ['mean', 'max']
    result = df.groupby('bulk_campaign_id').agg(agg)
    result.columns = [f'{prefix}_{col}_{stat}' for col, stat in result.columns]
    return result.reset_index()

campaign_features = build_campaign_features(full_campaign_daily_event, 'camp')
campaign_channel_features = build_campaign_features(full_campaign_daily_event_channel, 'camp_ch')

campaign_features = campaign_features.merge(campaign_channel_features, on='bulk_campaign_id', how='outer')
campaign_features.head()

In [ ]:
def build_client_campaign_features(message_features, chunksize=1_000_000):
    # Связь client_id -> bulk_campaign_id собирается отдельно, чтобы затем усреднить признаки кампаний по кампаниям клиента.
    pairs = []
    for chunk in iter_messages(chunksize=chunksize):
        pairs.append(chunk[['client_id', 'bulk_campaign_id']].drop_duplicates())
    pairs = pd.concat(pairs).drop_duplicates()
    client_campaign = pairs.merge(campaign_features, on='bulk_campaign_id', how='left')
    feature_cols = [c for c in client_campaign.columns if c not in ['client_id', 'bulk_campaign_id']]
    client_campaign_features = client_campaign.groupby('client_id')[feature_cols].mean().reset_index()
    return client_campaign_features

# Этот блок может выполняться несколько минут на полном apparel-messages.csv.
client_campaign_features = build_client_campaign_features(message_features, chunksize=1_000_000)
client_campaign_features.head()

## Объединение датасета для моделирования

In [ ]:
data = target.merge(purchase_features, on='client_id', how='left')
data = data.merge(message_features, on='client_id', how='left')
data = data.merge(client_campaign_features, on='client_id', how='left')

# Клиенты без истории в части таблиц получают нули в числовых признаках.
feature_cols = [c for c in data.columns if c not in ['client_id', 'target']]
data[feature_cols] = data[feature_cols].fillna(0)

print(data.shape)
display(data.head())

In [ ]:
X = data.drop(columns=['client_id', 'target'])
y = data['target']

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

X_train, X_valid, y_train, y_valid = train_test_split(
    X_train,
    y_train,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_train
)

print('train:', X_train.shape, y_train.mean().round(4))
print('valid:', X_valid.shape, y_valid.mean().round(4))
print('test:', X_test.shape, y_test.mean().round(4))

## Обучение базовых моделей

In [ ]:
numeric_features = X_train.columns.tolist()

numeric_preprocessor = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

preprocessor_scaled = ColumnTransformer(
    transformers=[('num', numeric_preprocessor, numeric_features)],
    remainder='drop'
)

preprocessor_tree = ColumnTransformer(
    transformers=[('num', SimpleImputer(strategy='median'), numeric_features)],
    remainder='drop'
)

models = {
    'LogisticRegression': Pipeline(steps=[
        ('preprocessor', preprocessor_scaled),
        ('model', LogisticRegression(max_iter=1000, class_weight='balanced', random_state=RANDOM_STATE))
    ]),
    'RandomForest': Pipeline(steps=[
        ('preprocessor', preprocessor_tree),
        ('model', RandomForestClassifier(
            n_estimators=300,
            max_depth=12,
            min_samples_leaf=5,
            class_weight='balanced',
            n_jobs=-1,
            random_state=RANDOM_STATE
        ))
    ]),
    'HistGradientBoosting': Pipeline(steps=[
        ('preprocessor', preprocessor_tree),
        ('model', HistGradientBoostingClassifier(
            learning_rate=0.05,
            max_iter=300,
            max_leaf_nodes=31,
            random_state=RANDOM_STATE
        ))
    ])
}

results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    valid_proba = model.predict_proba(X_valid)[:, 1]
    auc = roc_auc_score(y_valid, valid_proba)
    results.append({'model': name, 'valid_roc_auc': auc})
    print(f'{name}: ROC-AUC = {auc:.4f}')

results_df = pd.DataFrame(results).sort_values('valid_roc_auc', ascending=False)
results_df

## Подбор гиперпараметров лучшей sklearn-модели

Для максимизации ROC-AUC подбираем параметры для градиентного бустинга. `HistGradientBoostingClassifier` выбран как сильная модель из `sklearn`, которая хорошо работает с табличными данными.

In [ ]:
hgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_tree),
    ('model', HistGradientBoostingClassifier(random_state=RANDOM_STATE))
])

param_grid = {
    'model__learning_rate': [0.03, 0.05, 0.08],
    'model__max_iter': [200, 300],
    'model__max_leaf_nodes': [15, 31, 63],
    'model__l2_regularization': [0.0, 0.1]
}

grid = GridSearchCV(
    hgb_pipeline,
    param_grid=param_grid,
    scoring='roc_auc',
    cv=3,
    n_jobs=-1,
    verbose=1
)

grid.fit(pd.concat([X_train, X_valid]), pd.concat([y_train, y_valid]))

print('Лучшие параметры:', grid.best_params_)
print('CV ROC-AUC:', round(grid.best_score_, 4))

## Финальное тестирование

In [ ]:
best_model = grid.best_estimator_
test_proba = best_model.predict_proba(X_test)[:, 1]
test_pred = (test_proba >= 0.5).astype(int)

test_auc = roc_auc_score(y_test, test_proba)
print(f'Test ROC-AUC: {test_auc:.4f}')
print('
Classification report:')
print(classification_report(y_test, test_pred))
print('
Confusion matrix:')
print(confusion_matrix(y_test, test_pred))

## Важность признаков через permutation importance

У `HistGradientBoostingClassifier` нет встроенного `feature_importances_`, поэтому используем перестановочную важность признаков на тестовой выборке.

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring='roc_auc',
    n_repeats=5,
    random_state=RANDOM_STATE,
    n_jobs=-1
)

importance = pd.DataFrame({
    'feature': X_test.columns,
    'importance_mean': perm.importances_mean,
    'importance_std': perm.importances_std
}).sort_values('importance_mean', ascending=False)

importance.head(20)

In [ ]:
ax = importance.head(20).sort_values('importance_mean').plot(
    kind='barh',
    x='feature',
    y='importance_mean',
    figsize=(10, 8),
    legend=False,
    title='Топ-20 признаков по permutation importance'
)
ax.set_xlabel('Снижение ROC-AUC при перемешивании признака');

## Вывод

В проекте были подготовлены признаки по покупкам, активности в рассылках и агрегатам рекламных кампаний. Для борьбы с дисбалансом классов использовано стратифицированное разбиение, а качество моделей сравнивалось по ROC-AUC. Финальная модель выбирается по результатам кросс-валидации и затем проверяется на отложенной тестовой выборке.

Полученный пайплайн можно использовать для скоринга клиентов: модель возвращает вероятность покупки в ближайшие 90 дней, после чего маркетинг может выбрать клиентов с наибольшей вероятностью для персональных предложений.